# Output-Based

Metrik:
- Non-empty rate
- Mean_k, Median_k
- Unique terms
- Multiword rate
- Shannon entropy
- HHI (Herfindahl–Hirschman Index)
- In-text hit rate
- Context-fit embeddings (cosine similarity)

## 1) Setup

In [ ]:
# ============================================================
# 0) Optional: mount Google Drive (Colab only)
# ============================================================
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount("/content/drive")
#     print("[OK] Drive mounted.")
# except Exception:
#     print("[SKIP] Not running on Colab (no Drive mount).")

# ============================================================
# 1) Setup & Imports
# ============================================================
import os
import re
import ast
import math
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ============================================================
# 2) Plot styles + robustness directory (safe)
# ============================================================
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "font.size": 9,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
print("[OK] Matplotlib global styles configured.")

# Create robustness directory ONLY if OUT_DIR exists
FIG_ROBUST_DIR = None
if "OUT_DIR" in globals():
    FIG_ROBUST_DIR = OUT_DIR / "figures_intrinsic_robust"
    FIG_ROBUST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[OK] Robustness figures dir: {FIG_ROBUST_DIR.resolve()}")
else:
    print("[WARN] OUT_DIR not defined yet. Run Configuration cell first.")


def _pick_existing_col(df: pd.DataFrame, candidates):
    """Find a column in df that matches one of candidates (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        lc = cand.lower()
        if lc in lower_map:
            return lower_map[lc]
    return None

def enrich_dfs_with_metadata(dfs: dict, method_specs: list, base_dir: Path, verbose: bool = True) -> dict:
    """
    Enrich dfs[method] with metadata columns (comment_id, seg_id, source, date/year)
    pulled from the original files, aligned by row index.
    """
    for spec in method_specs:
        method_name = spec.get("method")
        rel_path = spec.get("path")
        if (not method_name) or (not rel_path) or (method_name not in dfs):
            continue

        original_path = base_dir / rel_path
        try:
            original_full_df = pd.read_csv(original_path)
        except FileNotFoundError:
            if verbose:
                print(f"[Meta] File not found for {method_name}: {original_path} -> skip")
            continue

        current = dfs[method_name].copy()

        meta_map = {}
        if "comment_id" not in current.columns:
            ccol = _pick_existing_col(original_full_df, ["comment_id","commentId","commentID","commentid"])
            if ccol: meta_map["comment_id"] = ccol
        if "seg_id" not in current.columns:
            scol = _pick_existing_col(original_full_df, ["seg_id","segId","segID","segid","seg_idx","seg_index","segment_id"])
            if scol: meta_map["seg_id"] = scol
        if "source" not in current.columns:
            sc = _pick_existing_col(original_full_df, ["source"])
            if sc: meta_map["source"] = sc
        if "date" not in current.columns:
            dc = _pick_existing_col(original_full_df, ["date"])
            if dc: meta_map["date"] = dc
        if ("year" not in current.columns) and ("date" not in current.columns):
            yc = _pick_existing_col(original_full_df, ["year"])
            if yc: meta_map["year"] = yc

        if not meta_map:
            dfs[method_name] = current
            continue

        # Align by row index (assumes processed df kept ordering)
        try:
            aligned = original_full_df.loc[current.index, list(meta_map.values())]
        except Exception as e:
            if verbose:
                print(f"[Meta] Index alignment failed for {method_name}: {e} -> skip")
            dfs[method_name] = current
            continue

        for std_name, raw_name in meta_map.items():
            current[std_name] = aligned[raw_name].values

        # types
        if "comment_id" in current.columns:
            current["comment_id"] = current["comment_id"].fillna("").astype(str)
        if "seg_id" in current.columns:
            current["seg_id"] = current["seg_id"].fillna("").astype(str)

        # derive year from date if present
        if "date" in current.columns:
            dt = pd.to_datetime(current["date"], errors="coerce")
            year_tmp = dt.dt.year
            nat_mask = year_tmp.isna()
            if nat_mask.any():
                date_str = current.loc[nat_mask, "date"].astype(str)
                extracted = date_str.str.extract(r"\b((?:19|20)\d{2})\b", expand=False)
                year_tmp.loc[nat_mask] = pd.to_numeric(extracted, errors="coerce")
            current["year"] = pd.to_numeric(year_tmp, errors="coerce").fillna(0).astype(int)

        if "year" in current.columns:
            current["year"] = pd.to_numeric(current["year"], errors="coerce").fillna(0).astype(int)

        dfs[method_name] = current
        if verbose:
            print(f"[Meta] Enriched {method_name}: added {list(meta_map.keys())}")

    if verbose:
        print("[Meta] Finished metadata enrichment.")
    return dfs

# Safe execution: only run if prerequisites already exist
if all(name in globals() for name in ["dfs", "METHOD_SPECS", "BASE_DIR"]):
    dfs = enrich_dfs_with_metadata(dfs=dfs, method_specs=METHOD_SPECS, base_dir=BASE_DIR, verbose=True)
else:
    print("[SKIP] Metadata enrichment not executed. Run cells in order:")
    print("       (1) Configuration -> defines BASE_DIR, OUT_DIR, METHOD_SPECS")
    print("       (2) Load data     -> defines dfs")


## 2) Configuration

In [ ]:
# ==== Configuration (local, reviewer-friendly) ====
from pathlib import Path
import os

# Data folder bundled in this share pack
BASE_DIR = Path("./data/dataset_baseline")

# Output folder for regenerated results
OUT_DIR = Path("./outputs/intrinsic_metrics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_SPECS = [
    {"method": "Rule-based",  "path": "rb_apect_terms_full_with_segtext.csv", "text_col": "seg_text",     "terms_col": "aspect_terms_rb_clean"},
    {"method": "Unsupervised","path": "unsup_terms_ranked_full_with_segtext.csv", "text_col": "seg_text", "terms_col": "aspect_terms_unsup"},
    {"method": "Hybrid",      "path": "dataset_hybrid_best_terms_sbert.csv", "text_col": "seg_text",     "terms_col": "hybrid_topk_terms"},

    # Baselines
    {"method": "LDA",         "path": "baseline_aspect_terms_lda_nmf_topk3.csv", "text_col": "seg_text_raw", "terms_col": "lda_terms"},
    {"method": "NMF",         "path": "baseline_aspect_terms_lda_nmf_topk3.csv", "text_col": "seg_text_raw", "terms_col": "nmf_terms"},
    {"method": "BTM",         "path": "baseline_aspect_terms_btm.csv",          "text_col": "seg_text_raw", "terms_col": "btm_terms"},
    {"method": "CTM",         "path": "dataset_baseline_CTM.csv",               "text_col": "seg_text_raw", "terms_col": "ctm_terms"},
    {"method": "BERTopic",    "path": "baseline_aspect_terms_bertopic.csv",     "text_col": "seg_text_raw", "terms_col": "bertopic_terms"},
    {"method": "Top2Vec",     "path": "baseline_aspect_terms_top2vec.csv",      "text_col": "seg_text_raw", "terms_col": "top2vec_terms"},
    {"method": "ABAE",        "path": "baseline_aspect_terms_abae_v2.csv",      "text_col": "seg_text_raw", "terms_col": "abae_terms"},
]

CACHE_DIR = Path("./outputs/_cache_eval_intrinsic")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR  =", BASE_DIR.resolve())
print("OUT_DIR   =", OUT_DIR.resolve())
print("CACHE_DIR =", CACHE_DIR.resolve())


## 3) Helper functions

In [ ]:
# --- Helpers: parsing & normalization ---

def is_nan(x):
    return x is None or (isinstance(x, float) and np.isnan(x))

def parse_terms_cell(x):
    if is_nan(x):
        return []
    if isinstance(x, list):
        return [str(t) for t in x if not is_nan(t)]
    if isinstance(x, (tuple, set)):
        return [str(t) for t in list(x) if not is_nan(t)]
    s = str(x).strip()
    if s == "" or s.lower() in {"nan","none","[]","kosong"}:
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t) for t in v if not is_nan(t)]
    except Exception:
        pass
    if s.startswith("[") and s.endswith("]"):
        s2 = s[1:-1].strip()
        if not s2:
            return []
        parts = re.split(r"\s*,\s*", s2)
        return [p.strip().strip("'\"") for p in parts if p.strip()]
    parts = re.split(r"\s*[,;]\s*", s)
    return [p.strip() for p in parts if p.strip()]

_ws_re = re.compile(r"\s+")
_nonword_keep_space = re.compile(r"[^\w\s]+", flags=re.UNICODE)

def norm_text(txt):
    if is_nan(txt):
        return ""
    t = str(txt).lower().replace("_", " ")
    t = _nonword_keep_space.sub(" ", t)
    t = _ws_re.sub(" ", t).strip()
    return t

def norm_term(term):
    return norm_text(term)

def token_count(term_norm):
    return len(term_norm.split()) if term_norm else 0

def shannon_entropy_from_counts(counts: Counter):
    total = sum(counts.values())
    if total == 0:
        return 0.0
    ent = 0.0
    for c in counts.values():
        p = c / total
        if p > 0:
            ent -= p * math.log(p, 2)
    return float(ent)

def hhi_from_counts(counts: Counter):
    total = sum(counts.values())
    if total == 0:
        return 0.0
    hhi = 0.0
    for c in counts.values():
        p = c / total
        hhi += p * p
    return float(hhi)

def in_text_hit(term_norm: str, text_norm: str):
    if not term_norm or not text_norm:
        return False
    if " " in term_norm:
        return term_norm in text_norm
    return re.search(rf"\b{re.escape(term_norm)}\b", text_norm) is not None

def calculate_all_intrinsic_metrics(df: pd.DataFrame, method_name: str):
    """
    Calculates all intrinsic metrics for a given DataFrame and method.
    Leverages existing metric calculation and caching functionalities.
    """
    # Get initial intrinsic metrics
    metrics = compute_intrinsic_metrics(df)

    # Conditionally calculate ContextFit
    if USE_CONTEXT_FIT:
        try:
            context_fit_val = compute_context_fit_for_method(df, method_name)
            metrics["ContextFit"] = context_fit_val
        except Exception as e:
            print(f"ContextFit SKIP for {method_name} -> {e}")
            metrics["ContextFit"] = np.nan
    else:
        metrics["ContextFit"] = np.nan

    return metrics

print("Defined function `calculate_all_intrinsic_metrics`.")

## 4) Load data (all methods)

In [ ]:
# --- Load all methods (unified schema) ---
def _pick_existing_col(df: pd.DataFrame, candidates):
    """Return first matching column name from candidates (case-insensitive), else None."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        lc = cand.lower()
        if lc in lower_map:
            return lower_map[lc]
    return None

def load_method_df(spec):
    path = BASE_DIR / spec["path"]
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    df = pd.read_csv(path)

    txt_col = spec["text_col"]
    trm_col = spec["terms_col"]
    if txt_col not in df.columns:
        raise KeyError(f"[{spec['method']}] text_col '{txt_col}' not in columns")
    if trm_col not in df.columns:
        raise KeyError(f"[{spec['method']}] terms_col '{trm_col}' not in columns")

    # Optional identifier columns (for seg_key)
    comment_col = _pick_existing_col(df, ["comment_id","commentId","commentID","commentid"])
    segid_col    = _pick_existing_col(df, ["seg_id","segId","segID","segid","seg_idx","seg_index","segment_id"])

    keep_cols = [txt_col, trm_col]
    if comment_col: keep_cols.append(comment_col)
    if segid_col: keep_cols.append(segid_col)

    # Optional metadata that may already exist in some method outputs
    for meta in ["source","date","year"]:
        mc = _pick_existing_col(df, [meta])
        if mc and mc not in keep_cols:
            keep_cols.append(mc)

    out = df[keep_cols].copy()

    # Standardize column names
    out.rename(columns={txt_col: "seg_text", trm_col: "terms_raw"}, inplace=True)
    if comment_col: out.rename(columns={comment_col: "comment_id"}, inplace=True)
    if segid_col:   out.rename(columns={segid_col: "seg_id"}, inplace=True)
    if "Source" in out.columns and "source" not in out.columns:
        out.rename(columns={"Source":"source"}, inplace=True)

    # Ensure types
    out["seg_text"] = out["seg_text"].fillna("").astype(str)
    if "comment_id" in out.columns:
        out["comment_id"] = out["comment_id"].astype(str).fillna("")
    if "seg_id" in out.columns:
        # keep as string to avoid losing leading zeros, but ensure stable
        out["seg_id"] = out["seg_id"].astype(str).fillna("")

    # Parse & normalize term lists
    out["terms_list"] = out["terms_raw"].apply(parse_terms_cell)
    out["terms_list_norm"] = out["terms_list"].apply(lambda lst: [norm_term(t) for t in lst if norm_term(t)])
    out["k"] = out["terms_list_norm"].apply(len).astype(int)

    # Normalize segment text (used in fallback seg_key and some metrics)
    out["seg_text_norm"] = out["seg_text"].apply(norm_text)

    # If 'year' exists but is not int, coerce
    if "year" in out.columns:
        out["year"] = pd.to_numeric(out["year"], errors="coerce").fillna(0).astype(int)

    return out

dfs = {}
for spec in METHOD_SPECS:
    try:
        dfs[spec["method"]] = load_method_df(spec)
        print(f"Loaded: {spec['method']:<15} rows={len(dfs[spec['method']]):,} cols={list(dfs[spec['method']].columns)[:8]}...")
    except Exception as e:
        print(f"SKIP:   {spec['method']:<15} -> {e}")

list(dfs.keys())

## 5) Fairness protocol and sanity checks

=== Fairness protocol implementation ===

(1) Ablation (UNSUP/RB/HYBRID): coverage-based fairness
- Each method is evaluated on its own segment (not forced to be equal).
- Report N_seg and NonEmpty; other metrics are read along with coverage information.
  
(2) Baseline benchmarking: baseline-aligned fairness
- All baselines are evaluated on a set of segments that are IDENTICAL to HYBRID-ATE.
- Use seg_key for cross-file/method matching.
- If a baseline has no output on a given segment, the segment is retained
and recorded as an empty output (k=0).

In [ ]:
ABLATION_METHODS = ["Rule-based", "Unsupervised", "Hybrid"]
BASELINE_METHODS = [m for m in dfs.keys() if m not in ABLATION_METHODS]

def build_seg_key(df: pd.DataFrame) -> pd.Series:
    """Segment key for cross-method alignment.

    Priority:
      1) comment_id + seg_id (jika tersedia dan tidak kosong)
      2) source||year||seg_text_norm (jika tersedia)
      3) seg_text_norm (fallback terakhir)
    """
    # 1) comment_id + seg_id
    if {"comment_id","seg_id"}.issubset(df.columns):
        c = df["comment_id"].fillna("").astype(str)
        s = df["seg_id"].fillna("").astype(str)
        has_both = (c.str.len() > 0) & (s.str.len() > 0)
        key = pd.Series([""]*len(df), index=df.index, dtype="object")
        key.loc[has_both] = c.loc[has_both] + "||" + s.loc[has_both]
        # fill remaining by fallback levels
    else:
        key = pd.Series([""]*len(df), index=df.index, dtype="object")
        has_both = pd.Series([False]*len(df), index=df.index)

    # 2) source||year||seg_text_norm
    need = ~has_both
    if need.any() and {"source","year","seg_text_norm"}.issubset(df.columns):
        key.loc[need] = (
            df.loc[need, "source"].fillna("").astype(str)
            + "||" + df.loc[need, "year"].fillna(0).astype(int).astype(str)
            + "||" + df.loc[need, "seg_text_norm"].fillna("").astype(str)
        )
        filled = key.str.len() > 0
        need = ~filled

    # 3) seg_text_norm only
    if need.any() and "seg_text_norm" in df.columns:
        key.loc[need] = df.loc[need, "seg_text_norm"].fillna("").astype(str)

    # Final: ensure non-null string
    key = key.fillna("").astype(str)
    return key

# Attach seg_key to each df (do not drop rows; keep for coverage-aware evaluation)
for m, df in dfs.items():
    dfx = df.copy()
    dfx["seg_key"] = build_seg_key(dfx)
    dfs[m] = dfx

# --- Track A: Ablation (as-is; coverage-aware reporting downstream)
dfs_ablation = {m: dfs[m] for m in ABLATION_METHODS if m in dfs}

# --- Track B: Baselines aligned to HYBRID segment set
dfs_baseline_aligned = {}
if "Hybrid" in dfs:
    hybrid_base = dfs["Hybrid"].copy()

    # Canonical segment frame = unique seg_key from Hybrid with its metadata/text
    keep_cols = ["seg_key", "seg_text", "seg_text_norm"]
    for col in ["comment_id","seg_id","source","year","date"]:
        if col in hybrid_base.columns and col not in keep_cols:
            keep_cols.append(col)

    hybrid_frame = hybrid_base[keep_cols].drop_duplicates("seg_key").reset_index(drop=True)

    # Helper to coerce missing list/k to empty
    def _ensure_terms_cols(frame: pd.DataFrame) -> pd.DataFrame:
        frame["terms_raw"] = frame["terms_raw"].fillna("")
        for lc in ["terms_list","terms_list_norm"]:
            if lc in frame.columns:
                frame[lc] = frame[lc].apply(lambda x: x if isinstance(x, list) else [])
            else:
                frame[lc] = [[] for _ in range(len(frame))]
        if "k" in frame.columns:
            frame["k"] = pd.to_numeric(frame["k"], errors="coerce").fillna(0).astype(int)
        else:
            frame["k"] = frame["terms_list_norm"].apply(len).astype(int)
        if "seg_text_norm" not in frame.columns:
            frame["seg_text_norm"] = frame["seg_text"].apply(norm_text)
        return frame

    # Align Hybrid to canonical frame (dedupe)
    hyb_terms = hybrid_base.drop_duplicates("seg_key")[["seg_key","terms_raw","terms_list","terms_list_norm","k"]]
    dfs_baseline_aligned["Hybrid"] = _ensure_terms_cols(hybrid_frame.merge(hyb_terms, on="seg_key", how="left"))

    # Align each baseline to Hybrid segments: left-join ensures identical segment set
    for m in BASELINE_METHODS:
        dfm = dfs[m].drop_duplicates("seg_key").copy()
        dfm_terms = dfm[["seg_key","terms_raw","terms_list","terms_list_norm","k"]]
        merged = hybrid_frame.merge(dfm_terms, on="seg_key", how="left")
        merged = _ensure_terms_cols(merged)  # missing outputs => empty (k=0)
        dfs_baseline_aligned[m] = merged

else:
    print("WARNING: Hybrid not found in dfs; baseline alignment skipped (dfs_baseline_aligned empty).")

print("Ablation methods loaded:", list(dfs_ablation.keys()))
print("Baseline-aligned methods:", [m for m in dfs_baseline_aligned.keys() if m!='Hybrid'][:10], "...")
if "Hybrid" in dfs_baseline_aligned:
    print("N(Hybrid canonical) =", len(dfs_baseline_aligned["Hybrid"]))

# Sanity check: segment-key uniqueness & alignment
for m, df in dfs.items():  # raw loaded
    dup = int(df["seg_key"].duplicated().sum())
    print(f"{m:12s} raw N={len(df):6d} dup_seg_key={dup}")

hyb = dfs.get("Hybrid")
if hyb is not None:
    print("Hybrid duplicated seg_key ratio =", float(hyb["seg_key"].duplicated().mean()))

for m, df in dfs.items():  # dfs = raw-loaded (sebelum aligned)
    dup = int(df["seg_key"].duplicated().sum())
    print(f"{m:10s} raw N={len(df)} dup_seg_key={dup}")

# khusus hybrid:
hyb = dfs["Hybrid"]
print("Hybrid dup ratio =", hyb["seg_key"].duplicated().mean())

## 6) Compute Intrinsic metrics (label-free)

def compute_intrinsic_metrics(df: pd.DataFrame):
    N = len(df)
    k_arr = df["k"].to_numpy()

    non_empty = float(np.mean(k_arr > 0)) if N else 0.0
    mean_k = float(np.mean(k_arr)) if N else 0.0
    median_k = float(np.median(k_arr)) if N else 0.0

    all_terms = [t for lst in df["terms_list_norm"].tolist() for t in lst]
    counts = Counter(all_terms)

    unique_terms = int(len(counts))
    multiword_rate = float(np.mean([token_count(t) > 1 for t in all_terms])) if all_terms else 0.0

    entropy = shannon_entropy_from_counts(counts)
    hhi = hhi_from_counts(counts)

    hits_per_seg = []
    for text_norm, terms in zip(df["seg_text_norm"].tolist(), df["terms_list_norm"].tolist()):
        if not terms:
            hits_per_seg.append(np.nan)
            continue
        hits_per_seg.append(float(np.mean([in_text_hit(t, text_norm) for t in terms])))

    in_text = float(np.nanmean(hits_per_seg)) if np.any(~pd.isna(hits_per_seg)) else 0.0

    return dict(
        N_segments=int(N),
        NonEmpty=non_empty,
        Mean_k=mean_k,
        Median_k=median_k,
        UniqueTerms=unique_terms,
        MultiwordRate=multiword_rate,
        ShannonEntropy=entropy,
        HHI=hhi,
        InTextHitRate=in_text,
    )

rows = []
for method, df in dfs.items():
    m = compute_intrinsic_metrics(df)
    m["Method"] = method
    rows.append(m)

summary_df = pd.DataFrame(rows).set_index("Method").sort_index()
summary_df

## 7) Context-fit (cosine) features

In [ ]:
# --- Context-fit embeddings (cosine) ---
# ContextFit(t, s_i)=cos(e(t), e(s_i))
# Summary: mean cosine per segment (mean over terms), then mean on non-empty segments.

USE_CONTEXT_FIT = True

def cache_path(name):
    return CACHE_DIR / name

def get_sbert_model():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("distiluse-base-multilingual-cased-v2")

def compute_context_fit_for_method(df: pd.DataFrame, method: str, batch_size: int = 64):
    seg_texts = df["seg_text"].fillna("").astype(str).tolist()
    has_terms = (df["k"].to_numpy() > 0)
    terms_lists = df["terms_list_norm"].tolist()

    unique_terms = sorted({t for lst in terms_lists for t in lst})
    if not unique_terms:
        return 0.0

    seg_cache = cache_path(f"seg_emb__{method}.npy")
    term_cache = cache_path(f"term_emb__{method}.npy")
    meta_cache = cache_path(f"meta__{method}.json")

    model = get_sbert_model()

    seg_emb = None
    if seg_cache.exists() and meta_cache.exists():
        try:
            seg_emb = np.load(seg_cache)
            meta = json.loads(meta_cache.read_text(encoding="utf-8"))
            if int(meta.get("n_segments", -1)) != len(seg_texts):
                seg_emb = None
        except Exception:
            seg_emb = None

    if seg_emb is None:
        seg_emb = model.encode(seg_texts, batch_size=batch_size, show_progress_bar=True,
                               convert_to_numpy=True, normalize_embeddings=True)
        np.save(seg_cache, seg_emb)
        meta_cache.write_text(json.dumps({"n_segments": len(seg_texts)}), encoding="utf-8")

    term_emb = None
    if term_cache.exists():
        try:
            term_emb = np.load(term_cache)
            if term_emb.shape[0] != len(unique_terms):
                term_emb = None
        except Exception:
            term_emb = None

    if term_emb is None:
        term_emb = model.encode(unique_terms, batch_size=batch_size, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True)
        np.save(term_cache, term_emb)

    term2idx = {t:i for i,t in enumerate(unique_terms)}
    per_seg = np.full((len(df),), np.nan, dtype=np.float32)

    for i, (ok, tlst) in enumerate(zip(has_terms, terms_lists)):
        if (not ok) or (not tlst):
            continue
        idxs = [term2idx[t] for t in tlst if t in term2idx]
        if not idxs:
            continue
        svec = seg_emb[i]
        tmat = term_emb[idxs]
        per_seg[i] = float(np.mean(np.dot(tmat, svec)))  # normalized => cosine

    return float(np.nanmean(per_seg)) if np.any(~np.isnan(per_seg)) else 0.0

if USE_CONTEXT_FIT:
    cf = {}
    for method, df in dfs.items():
        try:
            val = compute_context_fit_for_method(df, method)
            cf[method] = val
            print(f"ContextFit: {method:<10} mean={val:.4f}")
        except Exception as e:
            cf[method] = np.nan
            print(f"ContextFit SKIP: {method:<10} -> {e}")
    summary_df["ContextFit"] = pd.Series(cf)
else:
    summary_df["ContextFit"] = np.nan

summary_df

## 8) Robustness visualization helpers

In [ ]:
def generate_heatmap(df_to_plot, metric, row_col, outname_prefix, fig_dir=FIG_ROBUST_DIR):
    if df_to_plot.empty:
        print(f"Skipping heatmap for {metric} by {row_col} - DataFrame is empty.")
        return

    # Prepare the pivot table
    try:
        pivot_table = df_to_plot.reset_index().pivot_table(
            index=row_col,
            columns='Method',
            values=metric
        )
    except KeyError as e:
        print(f"Skipping heatmap for {metric} by {row_col} - Missing column: {e}.")
        return
    except Exception as e:
        print(f"Error creating pivot table for {metric} by {row_col}: {e}.")
        return

    # Handle case where pivot table might be empty after filtering
    if pivot_table.empty:
        print(f"Skipping heatmap for {metric} by {row_col} - Pivot table is empty.")
        return

    # Determine annotation format based on metric
    if metric == "UniqueTerms" or metric == "N_segments":
        fmt = ".0f"
    elif metric == "HHI":
        fmt = ".4f"
    else:
        fmt = ".3f"

    # Plotting
    fig_w = 10.0 # Increased width for better method label readability
    fig_h = max(5.0, pivot_table.shape[0] * 0.5) # Adjust height based on number of rows
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)

    sns.heatmap(
        pivot_table,
        annot=True,
        fmt=fmt,
        cmap="viridis", # Choose a suitable colormap
        linewidths=.5,
        ax=ax
    )

    ax.set_title(f"{metric} by {row_col}")
    ax.set_ylabel(row_col)
    ax.set_xlabel("Method")

    plt.xticks(rotation=45, ha='right') # Rotate method labels for better visibility
    plt.yticks(rotation=0)

    # Save (PNG high DPI + PDF vector)
    safe_name = f"{outname_prefix}_{metric.lower()}"
    png_path = fig_dir / f"{safe_name}.png"
    pdf_path = fig_dir / f"{safe_name}.pdf"

    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.02)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.02)

    plt.show()
    plt.close(fig)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

print("Defined function `generate_heatmap`.")

## 9) Compute metrics (overall, by source, by year)

In [ ]:
# === Overall intrinsic metrics (two tracks) ===
# Track A: Ablation (RB/UNSUP/HYB) -> coverage-aware reporting (N_seg + NonEmpty)
ablation_overall_rows = []
for method, df in dfs_ablation.items():
    print(f"[Ablation] Calculating overall intrinsic metrics for {method}...")
    metrics = calculate_all_intrinsic_metrics(df, method)
    metrics["N_seg"] = int(len(df))
    metrics["Method"] = method
    ablation_overall_rows.append(metrics)

ablation_overall_df = pd.DataFrame(ablation_overall_rows).set_index("Method")
ablation_overall_df = ablation_overall_df.loc[[m for m in ABLATION_METHODS if m in ablation_overall_df.index]]

ablation_csv_path = OUT_DIR / "intrinsic_overall_ablation_3methods.csv"
ablation_overall_df.to_csv(ablation_csv_path, index=True)
print(f"Saved: {ablation_csv_path}")
display(ablation_overall_df)

# Track B: Baseline comparison (Hybrid + baselines), aligned to Hybrid segment set (same N across methods)
baseline_overall_rows = []
for method, df in dfs_baseline_aligned.items():
    print(f"[Baseline-aligned] Calculating overall intrinsic metrics for {method}...")
    metrics = calculate_all_intrinsic_metrics(df, method)
    metrics["N_seg"] = int(len(df))
    metrics["Method"] = method
    baseline_overall_rows.append(metrics)

baseline_overall_df = pd.DataFrame(baseline_overall_rows).set_index("Method").sort_index()
baseline_csv_path = OUT_DIR / "intrinsic_overall_baseline_aligned_to_hybrid.csv"
baseline_overall_df.to_csv(baseline_csv_path, index=True)
print(f"Saved: {baseline_csv_path}")
display(baseline_overall_df)

In [ ]:
print("## Calculate Intrinsic Metrics by Source (two tracks)")

def calc_by_source(dfs_dict, tag: str):
    rows = []
    for method, df in dfs_dict.items():
        if 'source' not in df.columns:
            print(f"Skipping {tag} by-source for {method}: missing 'source'")
            continue
        for source in df['source'].dropna().unique():
            sub = df[df['source']==source].copy()
            if sub.empty:
                continue
            metrics = calculate_all_intrinsic_metrics(sub, f"{method}_{source}_{tag}")
            metrics['N_seg'] = int(len(sub))
            metrics['Method'] = method
            metrics['Source'] = source
            rows.append(metrics)
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows).set_index(['Method','Source']).sort_index()
    out_path = OUT_DIR / f"intrinsic_by_source_{tag}.csv"
    out.to_csv(out_path)
    print(f"Saved: {out_path}")
    return out

by_source_ablation_df = calc_by_source(dfs_ablation, tag='ablation_3methods')
display(by_source_ablation_df.head(20) if not by_source_ablation_df.empty else by_source_ablation_df)

by_source_baseline_df = calc_by_source(dfs_baseline_aligned, tag='baseline_aligned_to_hybrid')
display(by_source_baseline_df.head(20) if not by_source_baseline_df.empty else by_source_baseline_df)

## 10) Export tables (CSV/XLSX)

In [ ]:
### Save Evaluation Summary to CSV and XLSX

# === Pretty tables + XLSX export (post-refactor) ===
def _pretty(df, round_n=4):
    if df is None or len(df)==0:
        return df
    out = df.copy()
    # preferred column order
    col_order = []
    for c in ['N_seg','NonEmpty','Mean_k','Median_k','UniqueTerms','MultiwordRate','ShannonEntropy','HHI','InTextHitRate','ContextFit']:
        if c in out.columns:
            col_order.append(c)
    other_cols = [c for c in out.columns if c not in col_order]
    out = out[col_order + other_cols]
    for c in out.columns:
        if c in ['N_seg','UniqueTerms']:
            continue
        out[c] = pd.to_numeric(out[c], errors='coerce')

    float_cols = [c for c in out.columns if c not in ['N_seg','UniqueTerms']]
    out[float_cols] = out[float_cols].round(round_n)
    return out

ablation_pretty = _pretty(ablation_overall_df)
baseline_pretty = _pretty(baseline_overall_df)

print('Ablation (coverage-aware):')
display(ablation_pretty)

print('Baseline comparison (aligned to Hybrid segments):')
display(baseline_pretty)

# Save XLSX with multiple sheets
xlsx_path = OUT_DIR / 'intrinsic_evaluation_refactor_v2.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    if ablation_pretty is not None and len(ablation_pretty)>0:
        ablation_pretty.to_excel(writer, sheet_name='overall_ablation_3m')
    if baseline_pretty is not None and len(baseline_pretty)>0:
        baseline_pretty.to_excel(writer, sheet_name='overall_baseline_aligned')
    if 'by_source_ablation_df' in globals() and by_source_ablation_df is not None and len(by_source_ablation_df)>0:
        _pretty(by_source_ablation_df).to_excel(writer, sheet_name='by_source_ablation_3m')
    if 'by_source_baseline_df' in globals() and by_source_baseline_df is not None and len(by_source_baseline_df)>0:
        _pretty(by_source_baseline_df).to_excel(writer, sheet_name='by_source_baseline_aligned')
print(f'Saved XLSX: {xlsx_path}')

## 11) Robustness heatmaps/plots

In [ ]:
def generate_heatmap(df_to_plot, metric, row_col, outname_prefix, fig_dir=FIG_ROBUST_DIR):
    if df_to_plot.empty:
        print(f"Skipping heatmap for {metric} by {row_col} - DataFrame is empty.")
        return

    # Prepare the pivot table
    try:
        pivot_table = df_to_plot.reset_index().pivot_table(
            index=row_col,
            columns='Method',
            values=metric
        )
    except KeyError as e:
        print(f"Skipping heatmap for {metric} by {row_col} - Missing column: {e}.")
        return
    except Exception as e:
        print(f"Error creating pivot table for {metric} by {row_col}: {e}.")
        return

    # Handle case where pivot table might be empty after filtering
    if pivot_table.empty:
        print(f"Skipping heatmap for {metric} by {row_col} - Pivot table is empty.")
        return

    # Determine annotation format based on metric
    if metric == "UniqueTerms" or metric == "N_segments":
        fmt = ".0f"
    elif metric == "HHI":
        fmt = ".4f"
    else:
        fmt = ".3f"

    # Plotting
    fig_w = 10.0 # Increased width for better method label readability
    fig_h = max(5.0, pivot_table.shape[0] * 0.5) # Adjust height based on number of rows
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)

    sns.heatmap(
        pivot_table,
        annot=True,
        fmt=fmt,
        cmap="viridis", # Choose a suitable colormap
        linewidths=.5,
        ax=ax
    )

    ax.set_title(f"{metric} by {row_col}")
    ax.set_ylabel(row_col)
    ax.set_xlabel("Method")

    plt.xticks(rotation=45, ha='right') # Rotate method labels for better visibility
    plt.yticks(rotation=0)

    # Save (PNG high DPI + PDF vector)
    safe_name = f"{outname_prefix}_{metric.lower()}"
    png_path = fig_dir / f"{safe_name}.png"
    pdf_path = fig_dir / f"{safe_name}.pdf"

    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.02)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.02)

    plt.show()
    plt.close(fig)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

print("Defined function `generate_heatmap`.")